In [1]:
import torch
import torch.nn as nn
import urllib.request as req
import torch.nn.functional as F

Download Data

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [3]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [4]:
corpus = raw_text[:150]
print(corpus)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


Encode Data

In [5]:
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])
token_ids = encode(raw_text)
print("vocab_size", vocab_size)

vocab_size 65


In [6]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Parameters

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_dim_ttl = 128
mem_slots = 64
learning_rate = 1e-3
learning_rate_ttl = 2e-3
anchor_weight_ttl = 1e-2

Batch Data

In [8]:
block_size = 128
batch_size = 16
embedding_dim = 128

n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

def get_batch(device, split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Causal Attention Block

In [9]:
class CausalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = (Q @ K.transpose(-2, -1)) * (C ** -0.5)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, T, T)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        probs = F.softmax(scores, dim=-1)
        return probs @ V

Transformer Block

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, sf): #sf - scale factor
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalAttention(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * sf),
            nn.ReLU(),
            nn.Linear(dim * sf, dim)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

Simple GPT

In [11]:
class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, dim, sf, max_len): #sf - scale factor for dim
        super().__init__()
        self.vocab_size = vocab_size
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_len, dim)
        self.block = TransformerBlock(dim, sf)
        self.ln = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_vectors = self.tok_emb(idx)
        pos_vectors = self.pos_emb(torch.arange(T, device=idx.device))

        x = tok_vectors + pos_vectors
        x = self.block(x)
        x = self.ln(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))

        return logits, loss

Pretrain Base Model

In [12]:
base_model = SimpleGPT(vocab_size, embedding_dim, sf=4, max_len=block_size).to(device)
base_optimizer = torch.optim.AdamW(base_model.parameters(), lr=learning_rate)

base_pretrain_steps = 3000
base_model.train()
for step in range(base_pretrain_steps):
    x, y = get_batch(device, "train")
    _, loss = base_model(x, y)

    base_optimizer.zero_grad()
    loss.backward()
    base_optimizer.step()

    if step % 50 == 0:
        print(f"step {step}: base loss {loss.item():.4f}")

base_model.eval()

step 0: base loss 4.2836
step 50: base loss 2.7193
step 100: base loss 2.6217
step 150: base loss 2.6009
step 200: base loss 2.5269
step 250: base loss 2.5203
step 300: base loss 2.4453
step 350: base loss 2.4841
step 400: base loss 2.3183
step 450: base loss 2.3355
step 500: base loss 2.2251
step 550: base loss 2.1848
step 600: base loss 2.1775
step 650: base loss 2.1701
step 700: base loss 2.1677
step 750: base loss 2.1799
step 800: base loss 2.0921
step 850: base loss 2.0722
step 900: base loss 2.0259
step 950: base loss 2.0055
step 1000: base loss 2.0227
step 1050: base loss 1.9731
step 1100: base loss 2.0335
step 1150: base loss 1.9562
step 1200: base loss 1.9423
step 1250: base loss 1.9589
step 1300: base loss 1.9397
step 1350: base loss 1.9768
step 1400: base loss 1.8721
step 1450: base loss 1.8985
step 1500: base loss 1.8543
step 1550: base loss 1.8576
step 1600: base loss 1.8623
step 1650: base loss 1.8812
step 1700: base loss 1.8803
step 1750: base loss 1.9492
step 1800: base

SimpleGPT(
  (tok_emb): Embedding(65, 128)
  (pos_emb): Embedding(128, 128)
  (block): TransformerBlock(
    (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (attn): CausalAttention(
      (q): Linear(in_features=128, out_features=128, bias=False)
      (k): Linear(in_features=128, out_features=128, bias=False)
      (v): Linear(in_features=128, out_features=128, bias=False)
    )
    (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (ffn): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=128, bias=True)
    )
  )
  (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=128, out_features=65, bias=True)
)

Test Time Memory Module

In [13]:
class TTMemory(nn.Module):
    def __init__(self, base_model: nn.Module, base_dim=embedding_dim, feature_dim=feature_dim_ttl, mem_slots=mem_slots,
                 lr=learning_rate_ttl, anchor_weight=anchor_weight_ttl, pretrain_lr=learning_rate):
        super().__init__()
        self.base_model = base_model
        self.base_dim = base_dim
        self.feature_dim = feature_dim
        self.mem_slots = mem_slots
        self.lr = lr
        self.anchor_weight = anchor_weight

        for param in self.base_model.parameters():
            param.requires_grad = False

        self.base_model.eval()

        self.up_proj = nn.Linear(base_dim, feature_dim)
        self.down_proj = nn.Linear(feature_dim, base_dim)

        self.mem_keys = nn.Parameter(torch.randn(self.mem_slots, self.feature_dim) * 0.02)
        self.mem_values = nn.Parameter(torch.randn(self.mem_slots, self.feature_dim) * 0.02)
        self.register_buffer("init_keys", self.mem_keys.clone().detach())
        self.register_buffer("init_values", self.mem_values.clone().detach())

        # Slow weights: trained offline with real labels.
        self.pretrain_optimizer = torch.optim.AdamW(
            list(self.up_proj.parameters()) + list(self.down_proj.parameters()) + [self.mem_keys, self.mem_values],
            lr=pretrain_lr,
        )
        # Fast weights: adapted online at test time via the surprise signal.
        self.optimizer = torch.optim.SGD([self.mem_keys, self.mem_values], lr=self.lr, momentum=0.9)

    def _read_memory(self, features):
        norm_feat = F.normalize(features, p=2, dim=1)
        norm_keys = F.normalize(self.mem_keys, p=2, dim=1)

        similarity = norm_feat @ norm_keys.T
        attention = F.softmax(similarity, dim=-1)
        return attention @ self.mem_values

    def _compute_surprise_loss(self, outputs):
        probs = F.softmax(outputs, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=-1)
        return entropy.mean()

    def _anchor_loss(self):
        return self.anchor_weight * (
            F.mse_loss(self.mem_keys, self.init_keys) + F.mse_loss(self.mem_values, self.init_values)
        )

    def _augment(self, module, inputs, output):
        B, T, C = output.shape
        query = self.up_proj(output.reshape(B * T, C))
        read = self._read_memory(query)
        correction = self.down_proj(read).reshape(B, T, C)
        return output + correction

    def _run_augmented(self, x):
        handle = self.base_model.ln.register_forward_hook(self._augment)
        try:
            logits, _ = self.base_model(x)
        finally:
            handle.remove()
        return logits

    def pretrain_step(self, x, y):
        """Offline phase"""
        with torch.enable_grad():
            logits = self._run_augmented(x)
            task_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            self.pretrain_optimizer.zero_grad()
            task_loss.backward()
            self.pretrain_optimizer.step()

        return task_loss.item()

    def lock_pretrained(self):
        """Freeze the slow weights and re-anchor mem_keys/mem_values to their trained state."""
        for param in self.up_proj.parameters():
            param.requires_grad = False
        for param in self.down_proj.parameters():
            param.requires_grad = False
        self.init_keys.copy_(self.mem_keys.detach())
        self.init_values.copy_(self.mem_values.detach())

    def forward(self, x):
        """Online phase: no labels available, so adapt mem_keys/mem_values using the surprise signal only."""
        with torch.enable_grad():
            logits = self._run_augmented(x)
            surprise_loss = self._compute_surprise_loss(logits)
            loss = surprise_loss + self._anchor_loss()

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        return logits.detach(), surprise_loss.detach()

Pretrain Memory (Slow Weights)

In [14]:
ttmem = TTMemory(base_model).to(device)

mem_pretrain_steps = 1000
for step in range(mem_pretrain_steps):
    x, y = get_batch(device, "train")
    task_loss = ttmem.pretrain_step(x, y)

    if step % 50 == 0:
        print(f"step {step}: memory pretrain loss {task_loss:.4f}")

ttmem.lock_pretrained()

step 0: memory pretrain loss 1.6894
step 50: memory pretrain loss 1.7458
step 100: memory pretrain loss 1.7575
step 150: memory pretrain loss 1.8375
step 200: memory pretrain loss 1.7346
step 250: memory pretrain loss 1.8039
step 300: memory pretrain loss 1.6580
step 350: memory pretrain loss 1.6382
step 400: memory pretrain loss 1.7085
step 450: memory pretrain loss 1.7798
step 500: memory pretrain loss 1.7024
step 550: memory pretrain loss 1.7861
step 600: memory pretrain loss 1.6621
step 650: memory pretrain loss 1.7356
step 700: memory pretrain loss 1.6993
step 750: memory pretrain loss 1.7172
step 800: memory pretrain loss 1.6845
step 850: memory pretrain loss 1.7176
step 900: memory pretrain loss 1.7550
step 950: memory pretrain loss 1.7458


Test-Time Adaptation

In [15]:
for step in range(10):
    x, y = get_batch(device, "val")
    logits, surprise = ttmem(x)
    print(f"step {step}: surprise {surprise.item():.4f}")

step 0: surprise 1.7898
step 1: surprise 1.7709
step 2: surprise 1.8120
step 3: surprise 1.7812
step 4: surprise 1.7691
step 5: surprise 1.7923
step 6: surprise 1.8084
step 7: surprise 1.7974
step 8: surprise 1.7350
step 9: surprise 1.7871
